# Fundamental Weight Analysis

Scores each stock on 8 components (each ~[-10, 10]) and blends them into `Fundamental_Weight` (clipped to -10..+10).

| Component | Weight | Metrics |
|-----------|--------|---------|
| Valuation | 25% | Price_vs_FairValue × Valuation_Confidence |
| Profitability | 20% | TTM_ROE, TTM_ROA, TTM_NetProfitMargin |
| Growth | 15% | RevenueGrowth_YoY |
| Debt | 15% | Debt_to_Equity, DebtToAssets |
| Earnings Quality | 10% | EPSGrowth_YoY, NetIncomeGrowth_YoY |
| Efficiency | 5% | TTM_AssetTurnover, TTM_EquityTurnover |
| Margin Quality | 5% | TTM_GrossMargin, TTM_OperatingMargin |
| Return Quality | 5% | TTM_ROE, TTM_ROA |

- **Input**: `Reports/complete_company_analysis.xlsx` (sheet `2_Latest_Quarter_Complete`)
- **Output**: `Reports/balance_sheet_weights.csv` (read by `main_signal_analysis.ipynb`)

### Setup
Load the latest-quarter table written by `company_report_processing.ipynb`.

In [1]:
import os

import pandas as pd

import sector_mapping

REPORTS_DIR = sector_mapping.REPORTS_DIR  # absolute, next to sector_mapping.py
df = pd.read_excel(os.path.join(REPORTS_DIR, "complete_company_analysis.xlsx"), sheet_name="2_Latest_Quarter_Complete")
print(f"Loaded {len(df)} stocks")

Loaded 137 stocks


## 1. Component Scores

In [2]:
def valuation_score(pvf):
    """Price vs fair value (%): negative = undervalued."""
    if pd.isna(pvf):
        return 0
    if pvf < -20:
        return 10
    if pvf < -10:
        return 7
    if pvf < 10:
        return 3
    return -5


def growth_score(revenue_growth):
    if pd.isna(revenue_growth):
        return 0
    if revenue_growth > 20:
        return 10
    if revenue_growth > 10:
        return 7
    if revenue_growth > 0:
        return 3
    return -5


def debt_score(debt_to_equity, debt_to_assets):
    score = 0
    if pd.notna(debt_to_equity):
        if debt_to_equity < 0:  # negative shareholder equity -> worst bucket (old code scored it as "< 1.0 = great")
            score -= 5
        elif debt_to_equity < 1.0:
            score += 5
        elif debt_to_equity < 2.0:
            score += 3
        elif debt_to_equity < 3.0:
            score += 1
        elif debt_to_equity > 5.0:
            score -= 5
        else:
            score -= 2
    if pd.notna(debt_to_assets):
        if debt_to_assets < 30:
            score += 5
        elif debt_to_assets < 50:
            score += 3
        elif debt_to_assets < 70:
            score += 1
        elif debt_to_assets > 85:
            score -= 5
        else:
            score -= 2
    return max(-10, min(10, score))


def profitability_score(roe, roa, net_margin):
    score = 0
    if pd.notna(roe):
        if roe > 20:
            score += 4
        elif roe > 15:
            score += 3
        elif roe > 10:
            score += 2
        elif roe > 5:
            score += 1
        elif roe < 0:
            score -= 3
    if pd.notna(roa):
        if roa > 10:
            score += 3
        elif roa > 5:
            score += 2
        elif roa > 3:
            score += 1
        elif roa < 0:
            score -= 2
    if pd.notna(net_margin):
        if net_margin > 20:
            score += 3
        elif net_margin > 10:
            score += 2
        elif net_margin > 5:
            score += 1
        elif net_margin < 0:
            score -= 2
    return max(-10, min(10, score))


def efficiency_score(asset_turnover, equity_turnover):
    score = 0
    if pd.notna(asset_turnover):
        if asset_turnover > 1.5:
            score += 5
        elif asset_turnover > 1.0:
            score += 3
        elif asset_turnover > 0.5:
            score += 1
        elif asset_turnover < 0.2:
            score -= 3
        else:
            score -= 1
    if pd.notna(equity_turnover):
        if equity_turnover > 2.0:
            score += 5
        elif equity_turnover > 1.0:
            score += 3
        elif equity_turnover > 0.5:
            score += 1
        elif equity_turnover < 0.2:
            score -= 3
        else:
            score -= 1
    return max(-10, min(10, score))


def growth_points(growth):
    """Shared EPS / net income growth tiers."""
    if pd.isna(growth):
        return 0
    if growth > 25:
        return 5
    if growth > 15:
        return 3
    if growth > 5:
        return 2
    if growth > 0:
        return 1
    if growth < -10:
        return -5
    return -2


def earnings_quality_score(eps_growth, net_income_growth):
    return max(-10, min(10, growth_points(eps_growth) + growth_points(net_income_growth)))


def margin_quality_score(gross_margin, operating_margin):
    score = 0
    if pd.notna(gross_margin):
        if gross_margin > 50:
            score += 5
        elif gross_margin > 40:
            score += 3
        elif gross_margin > 30:
            score += 2
        elif gross_margin > 20:
            score += 1
        elif gross_margin < 10:
            score -= 3
        else:
            score -= 1
    if pd.notna(operating_margin):
        if operating_margin > 25:
            score += 5
        elif operating_margin > 15:
            score += 3
        elif operating_margin > 10:
            score += 2
        elif operating_margin > 5:
            score += 1
        elif operating_margin < 0:
            score -= 5
        else:
            score -= 2
    return max(-10, min(10, score))


def return_quality_score(roe, roa):
    score = 0
    if pd.notna(roe):
        if roe > 25:
            score += 5
        elif roe > 20:
            score += 4
        elif roe > 15:
            score += 3
        elif roe > 10:
            score += 2
        elif roe > 5:
            score += 1
        elif roe < 0:
            score -= 4
        else:
            score -= 1
    if pd.notna(roa):
        if roa > 15:
            score += 5
        elif roa > 10:
            score += 4
        elif roa > 5:
            score += 3
        elif roa > 3:
            score += 2
        elif roa > 1:
            score += 1
        elif roa < 0:
            score -= 4
        else:
            score -= 1
    return max(-10, min(10, score))

#### Apply the scores to every stock

In [3]:
def score(fn, *cols):
    return [fn(*vals) for vals in zip(*(df[c] for c in cols))]


# Valuation conviction scaled by fair-value method agreement (floor 0.25 when a fair value exists)
confidence = df["Valuation_Confidence"].fillna(0.6).clip(0, 1)
confidence = confidence.where(df["Price_vs_FairValue"].isna(), confidence.clip(lower=0.25))
df["Valuation_Score"] = (pd.Series(score(valuation_score, "Price_vs_FairValue")) * confidence).round(2)
df["Profitability_Score"] = score(profitability_score, "TTM_ROE", "TTM_ROA", "TTM_NetProfitMargin")
df["Growth_Score"] = score(growth_score, "RevenueGrowth_YoY")
df["Debt_Score"] = score(debt_score, "Debt_to_Equity", "DebtToAssets")
df["Earnings_Quality_Score"] = score(earnings_quality_score, "EPSGrowth_YoY", "NetIncomeGrowth_YoY")
df["Efficiency_Score"] = score(efficiency_score, "TTM_AssetTurnover", "TTM_EquityTurnover")
df["Margin_Quality_Score"] = score(margin_quality_score, "TTM_GrossMargin", "TTM_OperatingMargin")
df["Return_Quality_Score"] = score(return_quality_score, "TTM_ROE", "TTM_ROA")

## 2. Fundamental Weight & Export

In [4]:
WEIGHTS = {
    "Valuation_Score": 0.25, "Profitability_Score": 0.20, "Growth_Score": 0.15, "Debt_Score": 0.15,
    "Earnings_Quality_Score": 0.10, "Efficiency_Score": 0.05, "Margin_Quality_Score": 0.05, "Return_Quality_Score": 0.05,
}
df["Fundamental_Weight"] = sum(df[c] * w for c, w in WEIGHTS.items()).clip(-10, 10).round(2)

out = df[["Symbol", *WEIGHTS, "Fundamental_Weight"]]
out.to_csv(os.path.join(REPORTS_DIR, "balance_sheet_weights.csv"), index=False)
print(f"✅ Saved {len(out)} stocks to balance_sheet_weights.csv")
out.sort_values("Fundamental_Weight", ascending=False).head(10)

✅ Saved 137 stocks to balance_sheet_weights.csv


,Symbol,Valuation_Score,Profitability_Score,Growth_Score,Debt_Score,Earnings_Quality_Score,Efficiency_Score,Margin_Quality_Score,Return_Quality_Score,Fundamental_Weight
94,NVDA,2.03,10,10,10,10,8,10,10,7.91
51,GOOGL,2.21,10,10,10,10,4,10,10,7.75
80,MSFT,4.88,10,7,8,10,0,10,10,7.47
10,APP,6.06,10,10,1,10,6,10,10,7.46
30,CRM,6.82,8,7,8,10,0,8,6,7.26
7,AMZN,4.68,9,7,8,10,4,7,8,7.17
124,UBER,6.97,9,7,4,10,4,5,9,7.09
3,ADBE,6.25,10,7,4,3,6,10,10,6.81
9,APH,2.12,9,10,8,8,4,5,9,6.73
65,KLAC,6.50,10,3,2,10,6,10,10,6.68
